# Day 6 - Code breaks, block 1: attention

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the value marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

Today we code **little** and look a lot: code break 1 is attention with small numbers, code breaks 2 and 3 use a
real pre-trained Transformer (DistilBERT, about 250 MB, downloaded once).

## Setup - run these two cells first
The first cell loads PyTorch and DistilBERT - a small Transformer that was pre-trained on English Wikipedia and books.

In [ ]:
import matplotlib.pyplot as plt
import torch
from transformers import AutoModel, AutoTokenizer, logging, pipeline

logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased", output_attentions=True)
fill = pipeline("fill-mask", model="distilbert-base-uncased")
print("ready - PyTorch", torch.__version__)

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `VEC` - a tiny "dictionary" of words as vectors with 2 numbers: dimension 1 = nature, dimension 2 = finance
- `plot_bank(sentence, weights, new_bank)` - draws the word vectors and the new vector of "bank"
- `bert_attention(sentence)` - runs DistilBERT and returns the words and all attention weights
  (6 layers × 12 heads × words × words)
- `show_attention(words, att, word)` - draws the attention of one head as a picture and the row of one word

In [ ]:
VEC = {"the": [0.1, 0.1], "river": [2.0, 0.0], "money": [0.0, 2.0], "bank": [1.0, 1.0]}


def plot_bank(sentence, weights, new_bank):
    plt.figure(figsize=(4, 4))
    for w in sentence:
        v = VEC[w]
        plt.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="gray"))
        plt.text(v[0] + 0.05, v[1] + 0.05, w, color="gray")
    plt.annotate("", xy=new_bank.tolist(), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#EB811B", lw=3))
    plt.text(new_bank[0] + 0.05, new_bank[1] - 0.2, "new bank", color="#EB811B", fontweight="bold")
    plt.xlim(-0.2, 2.4); plt.ylim(-0.2, 2.4)
    plt.xlabel("dimension 1: nature"); plt.ylabel("dimension 2: finance")
    plt.show()


def bert_attention(sentence):
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        att = torch.stack(bert(**enc).attentions)[:, 0]          # (layers, heads, words, words)
    return tokenizer.convert_ids_to_tokens(enc["input_ids"][0]), att


def show_attention(words, att, word):
    i = words.index(word)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].imshow(att, cmap="Oranges")
    axes[0].set_xticks(range(len(words)), words, rotation=90); axes[0].set_yticks(range(len(words)), words)
    axes[0].set_xlabel("looks at"); axes[0].set_title("attention of one head")
    axes[1].barh(range(len(words)), att[i], color="#EB811B"); axes[1].invert_yaxis()
    axes[1].set_yticks(range(len(words)), words); axes[1].set_title(f'where does "{word}" look?')
    plt.tight_layout()
    plt.show()


print("helpers ready")

## 💻 Code break 1 - Attention by numbers: "river bank" or "money bank"?
The last word ("bank") compares itself with every word (dot product), softmax turns the scores into weights that add
up to 1, and the new "bank" is the weighted average of all word vectors.

In [ ]:
SENTENCE = ["the", "river", "bank"]       # 👈 try "money" instead of "river"

X = torch.tensor([VEC[w] for w in SENTENCE])      # one vector per word: (3 words, 2 numbers)
scores = X[-1] @ X.T                              # "bank" compares itself with every word "How similar is bank to the river? How similar is bank to money?"
weights = torch.softmax(scores, dim=0)            # numbers between 0 and 1 that add up to 1
new_bank = weights @ X                            # new vector for "bank" that is a weighted average of the other words
print("scores: ", [round(v, 2) for v in scores.tolist()])
print("weights:", [round(w, 2) for w in weights.tolist()], "  new bank:", [round(v, 2) for v in new_bank.tolist()])
plot_bank(SENTENCE, weights, new_bank)

🔧 **Change it:**
1. `SENTENCE = ["the", "money", "bank"]` - where does "bank" move now?
2. `SENTENCE = ["river", "money", "bank"]` - both!

<details><summary>👀 What you should see</summary>

- "the river bank": scores **[0.2, 2.0, 2.0]**, weights **[0.08, 0.46, 0.46]**, new bank **[1.39, 0.47]** - it moved
  towards "river" (nature)
- "the money bank": the same weights, new bank **[0.47, 1.39]** - it moved towards "money" (finance)
- "river money bank": weights **[0.33, 0.33, 0.33]**, new bank **[1.0, 1.0]** - the context does not decide, so the
  meaning stays in the middle
- this is all attention does: **every word becomes a weighted mix of the words around it**. A real Transformer
  learns the vectors (and three matrices for query, key and value) with backpropagation
</details>

## 💻 Code break 2 - Attention in a real Transformer
DistilBERT has 6 layers with 12 heads each - 72 attention pictures. Which word does "it" look at?

In [ ]:
SENTENCE = "The animal didn't cross the street because it was too tired."   # 👈 "... too wide."
LAYER = 4            # 👈 0 ... 5
HEAD = 0             # 👈 0 ... 11

words, att = bert_attention(SENTENCE)
print("words:", words)
print("attention shape:", tuple(att.shape), " = (layers, heads, words, words)")
show_attention(words, att[LAYER, HEAD], "it")

🔧 **Change it:**
1. `HEAD = 1`, then `LAYER = 0`. Do all heads look at "animal"?
2. Change "tired" to "wide" - now "it" should mean the street. What happens in layer 4, head 0?

<details><summary>👀 What you should see</summary>

- 16 words: `[CLS]` and `[SEP]` are special start/end tokens; "didn't" is split into "didn", "'", "t"
- attention shape **(6, 12, 16, 16)** - 6 layers, 12 heads, every word looks at all 16 words
- layer 4, head 0: "it" puts **0.61** of its attention on **"animal"** - nobody told the model; it learned this from text
- head 1 of layer 4 does the same (0.60). In layer 0 the heads look mostly at neighbouring words ("because", "was", "too") or at `[CLS]` -
  different heads do different jobs
- "too wide": attention on "animal" drops to **0.26**, on "street" it rises only from 0.02 to **0.09**. This small
  model does not fully "get" it - attention weights are a hint, not a proof of understanding
</details>

## 💻 Code break 3 - What did the pre-trained Transformer learn?
DistilBERT was trained by hiding words in millions of sentences and guessing them ("fill in the blank"). Here it guesses
the word behind `[MASK]` - in sentences it has never seen.

In [ ]:
TEXT = "The patient had an epileptic [MASK] during the night."     # 👈 your own sentence with one [MASK]

for guess in fill(TEXT, top_k=5):
    print(f"{guess['token_str']:12s} {guess['score']:.2f}")

🔧 **Change it:** write your own sentence - something from medicine, your hometown, or your studies.

<details><summary>👀 What you should see</summary>

- "epileptic [MASK]": **seizure 0.95** - the model learned this from text alone, no labels
- "The doctor looked at the [MASK] of the brain.": only weak guesses (contents 0.11, anatomy 0.05) - several words fit
- this is **pre-training**: learn from huge unlabelled data first. In the afternoon we do the second half -
  **transfer learning**: adapt a pre-trained model to our own small task
</details>